# 04 - Évaluation de BLIP et GIT

Ce notebook évalue les **jugements image-légende** produits par le notebook 03 :
mesures chiffrées, comparaison temps/mémoire, matrices de confusion et exemples visuels.
Il fonctionne sur **CPU**, sans recharger les modèles.

**Avant de commencer :** exécuter les notebooks 01, 02 et 03 avec le même Google Drive,
puis choisir **Exécution → Tout exécuter** dans Colab. Les CSV doivent être dans
`MyDrive/ia_visuelle/results/` : `baseline_alignment_pairs.csv`,
`judgements_{blip|git}_{fp16|int8|int4}.csv` et, pour les ressources,
`model_judgement_summary.csv`. Les configurations absentes seront signalées.

La comparaison avec les humains est calculée seulement si de vraies annotations sont fournies.

In [ ]:
# Installer seulement les bibliothèques manquantes.
import importlib.util
import subprocess
import sys

packages = {
    "numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib",
    "sklearn": "scikit-learn", "scipy": "scipy", "PIL": "pillow",
    "datasets": "datasets",
}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

In [ ]:
import os
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from PIL import Image
from scipy.stats import spearmanr
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, cohen_kappa_score,
)

# Même dossier Drive que les notebooks précédents ; en local, dossier du projet.
try:
    from google.colab import drive
except ImportError:
    local_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    PROJECT_DIR = Path(os.environ.get("VLM_PROJECT_DIR", str(local_root)))
else:
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/ia_visuelle")

DATA_DIR = PROJECT_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
AFFICHER_IMAGES = True  # False permet de calculer seulement les tableaux/graphiques.
# Deux modèles et trois précisions donnent six configurations à comparer.
CONFIGURATIONS = [(model, precision) for model in ("blip", "git")
                  for precision in ("fp16", "int8", "int4")]
PAIR_KEYS = ["image_id", "caption", "pair_type"]
print(f"Dossier des résultats : {RESULTS_DIR}")

## 1. Vérifier les paires et charger les décisions

Comme dans le notebook 03, **1 = légende correcte** et **0 = légende incorrecte**.
Pour chaque image, le modèle choisit la légende ayant la plus petite `loss`
parmi deux candidates. Nous conservons ses `predicted_label` enregistrés :
aucun seuil n'est ajusté sur les réponses attendues.

Cette évaluation mesure un **choix entre deux légendes**, pas une décision indépendante
sur une image seule. Les labels du notebook 02 sont construits automatiquement ;
ils ne sont pas des annotations humaines. Les égalités de `loss`, départagées selon
l'ordre des lignes dans le notebook 03, sont comptées pour rendre ce biais visible.

In [ ]:
# Vérifier les fichiers avant de commencer les calculs.
def read_csv_checked(path, required):
    if not path.exists():
        raise FileNotFoundError(
            f"Fichier absent : {path}. Exécutez le notebook qui le produit "
            "et vérifiez le dossier results de Google Drive."
        )
    frame = pd.read_csv(path)
    missing_columns = sorted(set(required) - set(frame.columns))
    if missing_columns:
        raise ValueError(f"{path.name} : colonnes manquantes {missing_columns}")
    if frame.empty:
        raise ValueError(f"{path.name} est vide.")
    return frame


pairs = read_csv_checked(
    RESULTS_DIR / "baseline_alignment_pairs.csv", PAIR_KEYS + ["label"]
)[PAIR_KEYS + ["label"]].copy()
if pairs.isna().any().any() or pairs["caption"].str.strip().eq("").any():
    raise ValueError("Les paires contiennent des valeurs vides.")
ids = pd.to_numeric(pairs["image_id"], errors="raise")
if (ids < 0).any() or (ids % 1 != 0).any():
    raise ValueError("image_id doit être un entier positif ou nul.")
pairs["image_id"] = ids.astype(int)
if not pairs["label"].isin([0, 1]).all():
    raise ValueError("Les labels doivent être 0 ou 1.")
if not pairs["pair_type"].map({"correct": 1, "incorrect": 0}).eq(pairs["label"]).all():
    raise ValueError("Les colonnes pair_type et label ne correspondent pas.")
if pairs.duplicated(PAIR_KEYS).any():
    raise ValueError("Certaines paires sont présentes plusieurs fois.")
# Chaque image doit avoir une légende correcte et une légende incorrecte.
counts = pairs.groupby("image_id")["label"].agg(["size", "sum"])
if not ((counts["size"] == 2) & (counts["sum"] == 1)).all():
    raise ValueError("Il faut une paire correcte et une incorrecte par image.")

# Identifiants reproductibles, mélangés pour ne pas révéler les labels aux humains.
pairs = pairs.sort_values(PAIR_KEYS).sample(frac=1, random_state=SEED).reset_index(drop=True)
pairs.insert(0, "pair_id", np.arange(len(pairs)))
print(f"{len(pairs)} paires pour {pairs['image_id'].nunique()} images.")

In [ ]:
experiments = {}
unavailable = []

for model, precision in CONFIGURATIONS:
    path = RESULTS_DIR / f"judgements_{model}_{precision}.csv"
    if not path.exists():
        unavailable.append(f"{model}/{precision}")
        continue

    # Reprendre les décisions sauvegardées par le notebook 03.
    required = PAIR_KEYS + ["label", "loss", "predicted_label"]
    saved = read_csv_checked(path, required)[required].copy()
    if saved.isna().any().any():
        raise ValueError(f"{path.name} : valeurs manquantes.")
    if not saved["predicted_label"].isin([0, 1]).all():
        raise ValueError(f"{path.name} : predicted_label doit être 0 ou 1.")
    saved["loss"] = pd.to_numeric(saved["loss"], errors="raise")
    if not np.isfinite(saved["loss"]).all() or (saved["loss"] < 0).any():
        raise ValueError(f"{path.name} : loss doit être finie et positive ou nulle.")

    # Joindre sur le contenu, jamais sur la position des lignes des fichiers.
    frame = pairs.merge(
        saved, on=PAIR_KEYS, how="outer", suffixes=("", "_saved"),
        validate="one_to_one", indicator=True,
    )
    if not frame["_merge"].eq("both").all():
        raise ValueError(f"{path.name} : paires manquantes ou étrangères au notebook 02.")
    if not frame["label"].eq(frame["label_saved"]).all():
        raise ValueError(f"{path.name} : labels différents du notebook 02.")
    if not frame.groupby("image_id")["predicted_label"].sum().eq(1).all():
        raise ValueError(f"{path.name} : il faut une seule légende choisie par image.")
    selected_loss = frame.loc[frame["predicted_label"].eq(1)].set_index("image_id")["loss"]
    minimum_loss = frame.groupby("image_id")["loss"].min()
    if not selected_loss.sort_index().eq(minimum_loss.sort_index()).all():
        raise ValueError(f"{path.name} : la légende choisie n'a pas la plus petite loss.")

    frame = frame.drop(columns=["label_saved", "_merge"])
    frame["model"], frame["precision"] = model, precision
    frame["score"] = -frame["loss"]  # Score de cohérence, pas une probabilité.
    frame["is_correct"] = frame["label"].eq(frame["predicted_label"])
    experiments[(model, precision)] = frame.sort_values("pair_id")

if unavailable:
    print("Configurations absentes :", ", ".join(unavailable))
if not experiments:
    raise FileNotFoundError(
        "Aucun fichier judgements_*.csv disponible. Exécutez le notebook 03."
    )
all_predictions = pd.concat(experiments.values(), ignore_index=True)
all_predictions.to_csv(RESULTS_DIR / "evaluation_predictions.csv", index=False)
print(f"{len(experiments)}/6 configurations chargées et vérifiées.")

## 2. Calculer les mesures

- **Accuracy** : proportion de décisions correctes ; la baseline aléatoire du notebook 02 vaut 50 %.
- **Précision, rappel et F1** : qualité des décisions pour la classe « légende correcte ».
- **AUC ROC** : classement des paires à partir du score `-loss`, sans seuil.

Les temps et le pic de mémoire GPU proviennent du notebook 03 et ne sont pas remesurés ici.
Le pic correspond à la mémoire allouée par PyTorch, pas à toute la VRAM de la machine.

In [ ]:
# Calculer les mesures pour chaque configuration disponible.
rows = []
for (model, precision), frame in experiments.items():
    # Comparer la réponse attendue à la décision du modèle.
    y_true = frame["label"].astype(int)
    y_pred = frame["predicted_label"].astype(int)
    rows.append({
        "model": model, "precision": precision,
        "images": frame["image_id"].nunique(), "pairs": len(frame),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_positive": precision_score(y_true, y_pred, zero_division=0),
        "recall_positive": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, frame["score"]),
        "tied_images": int(frame.groupby("image_id")["loss"].nunique().eq(1).sum()),
    })
metrics = pd.DataFrame(rows)

# Un choix au hasard entre deux légendes donne 50 % en moyenne.
baseline_accuracy = 0.5
baseline_path = RESULTS_DIR / "baseline_metrics.csv"
if baseline_path.exists():
    baseline = read_csv_checked(baseline_path, ["experiment", "accuracy"])
    random_baseline = baseline.loc[baseline["experiment"].eq("random_baseline"), "accuracy"]
    if len(random_baseline) != 1 or not np.isclose(float(random_baseline.iloc[0]), 0.5):
        raise ValueError("La baseline du notebook 02 doit avoir une accuracy théorique de 0,5.")
metrics["gain_over_random_pp"] = 100 * (metrics["accuracy"] - baseline_accuracy)

In [ ]:
summary_path = RESULTS_DIR / "model_judgement_summary.csv"
resource_columns = ["seconds", "seconds_per_pair", "peak_memory_mb"]

if summary_path.exists():
    summary = read_csv_checked(
        summary_path, ["model", "precision", "accuracy", "pairs"] + resource_columns
    )
    if summary.duplicated(["model", "precision"]).any():
        raise ValueError("Le résumé contient plusieurs lignes pour une même configuration.")
    for column in resource_columns + ["accuracy", "pairs"]:
        summary[column] = pd.to_numeric(summary[column], errors="raise")
        if not np.isfinite(summary[column]).all() or (summary[column] < 0).any():
            raise ValueError(f"Valeur invalide dans le résumé : {column}")

    metrics = metrics.merge(
        summary[["model", "precision", "accuracy", "pairs"] + resource_columns],
        on=["model", "precision"], how="left", suffixes=("", "_summary"),
        validate="one_to_one",
    )
    present = metrics["pairs_summary"].notna()
    if not metrics.loc[present, "pairs"].eq(metrics.loc[present, "pairs_summary"]).all():
        raise ValueError("Les nombres de paires du résumé ne correspondent pas aux CSV.")
    if not np.allclose(
        metrics.loc[present, "accuracy"], metrics.loc[present, "accuracy_summary"]
    ):
        raise ValueError("L'accuracy recalculée diffère du résumé du notebook 03.")
    metrics = metrics.drop(columns=["accuracy_summary", "pairs_summary"])
else:
    print("Résumé absent : les temps et la mémoire ne sont pas disponibles.")
    for column in resource_columns:
        metrics[column] = np.nan

# Utiliser la version fp16 de chaque modèle comme référence.
fp16 = metrics.loc[metrics["precision"].eq("fp16")].set_index("model")
metrics["accuracy_delta_vs_fp16_pp"] = 100 * (
    metrics["accuracy"] - metrics["model"].map(fp16["accuracy"])
)
reference_memory = metrics["model"].map(fp16["peak_memory_mb"]).replace(0, np.nan)
metrics["memory_saving_vs_fp16_pct"] = 100 * (
    1 - metrics["peak_memory_mb"] / reference_memory
)
metrics.to_csv(RESULTS_DIR / "evaluation_metrics.csv", index=False)
display(metrics.round(4))
print("pp = points de pourcentage ; NaN = mesure indisponible.")
if metrics["tied_images"].sum():
    print("Attention : des égalités de loss existent ; consulter la colonne tied_images.")

## 3. Comparer les configurations

La ligne pointillée représente la baseline aléatoire. Une valeur plus élevée est
préférable pour l'accuracy ; une valeur plus faible est préférable pour le temps et la mémoire.
Les configurations sont comparées sur exactement les mêmes paires.

In [ ]:
# Afficher la qualité, le temps de calcul et la mémoire côte à côte.
names = metrics["model"].str.upper() + " / " + metrics["precision"]
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
plots = [
    ("accuracy", "Accuracy", "Proportion correcte"),
    ("seconds_per_pair", "Temps par paire", "Secondes"),
    ("peak_memory_mb", "Pic de mémoire GPU", "Mo"),
]
for axis, (column, title, ylabel) in zip(axes, plots):
    valid = metrics[column].notna()
    if valid.any():
        axis.bar(np.flatnonzero(valid), metrics.loc[valid, column], color="#4776b4")
    else:
        axis.text(0.5, 0.5, "Mesure indisponible", ha="center", transform=axis.transAxes)
    axis.set_xticks(range(len(metrics)), names, rotation=45, ha="right")
    axis.set(title=title, ylabel=ylabel)
    axis.grid(axis="y", alpha=0.2)
axes[0].axhline(baseline_accuracy, color="red", linestyle="--", label="Hasard : 50 %")
axes[0].set_ylim(0, 1.05)
axes[0].legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "comparaison_configurations.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

# Montrer les bonnes et les mauvaises décisions pour chaque configuration.
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for axis, configuration in zip(axes.flat, CONFIGURATIONS):
    axis.set_title(f"{configuration[0].upper()} / {configuration[1]}")
    if configuration not in experiments:
        axis.text(0.5, 0.5, "Configuration absente", ha="center", transform=axis.transAxes)
        axis.axis("off")
        continue
    frame = experiments[configuration]
    matrix = confusion_matrix(frame["label"], frame["predicted_label"], labels=[0, 1])
    ConfusionMatrixDisplay(matrix, display_labels=["Incorrecte", "Correcte"]).plot(
        ax=axis, cmap="Blues", colorbar=False, values_format="d"
    )
    axis.set(xlabel="Décision du modèle", ylabel="Label automatique")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "matrices_confusion.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

## 4. Comparer avec des annotations humaines, si disponibles

Le fichier `human_annotations_template.csv` est créé dans `results/`.
En faire une copie nommée **`human_annotations.csv`**, puis annoter tout ou partie des paires :
`human_label` = 0 ou 1 ; `human_score` = note de cohérence de 1 à 5 (facultative).

Une ligne représente un jugement humain final par paire. Garder `pair_id`, `image_id`
et `caption` inchangés, consulter l'image correspondante et laisser les cases non évaluées
vides. Le modèle et le label automatique ne figurent pas dans ce modèle d'annotation.
Après annotation, réexécuter cette section.

L'accord et le kappa de Cohen comparent les décisions binaires ; Spearman compare
`-loss` aux notes humaines. Une corrélation indéfinie (notes constantes, par exemple)
reste vide. Sans annotations, aucune mesure humaine n'est inventée.

In [ ]:
# Préparer un fichier vide à remplir après avoir regardé les images.
template = pairs[["pair_id", "image_id", "caption"]].copy()
template["human_label"] = pd.NA
template["human_score"] = pd.NA
template_path = RESULTS_DIR / "human_annotations_template.csv"
template.to_csv(template_path, index=False)  # Ne remplace jamais human_annotations.csv.
print(f"Modèle d'annotation : {template_path}")

human_columns = [
    "model", "precision", "n_human_labels", "agreement", "cohen_kappa",
    "n_human_scores", "spearman_rho",
]
human_metrics = pd.DataFrame(columns=human_columns)
human_path = RESULTS_DIR / "human_annotations.csv"

if not human_path.exists():
    print("Annotations humaines absentes : comparaison à compléter après leur collecte.")
else:
    human = read_csv_checked(human_path, ["pair_id", "image_id", "caption"])
    if human.duplicated("pair_id").any():
        raise ValueError("Il faut une seule ligne finale par pair_id dans les annotations.")
    check = human.merge(
        pairs[["pair_id", "image_id", "caption"]], on="pair_id", how="left",
        suffixes=("", "_expected"), validate="one_to_one", indicator=True,
    )
    if (not check["_merge"].eq("both").all()
            or not check["image_id"].eq(check["image_id_expected"]).all()
            or not check["caption"].eq(check["caption_expected"]).all()):
        raise ValueError("Les annotations ne correspondent pas aux paires de cette expérience.")

    for column, allowed in [("human_label", [0, 1]), ("human_score", [1, 2, 3, 4, 5])]:
        if column not in human:
            human[column] = np.nan
        human[column] = pd.to_numeric(human[column], errors="raise")
        if not human[column].dropna().isin(allowed).all():
            raise ValueError(f"Valeur invalide pour {column} : valeurs permises {allowed}.")

    human_rows = []
    for (model, precision), frame in experiments.items():
        # Associer chaque annotation humaine à la bonne paire image-légende.
        compared = frame.merge(
            human[["pair_id", "human_label", "human_score"]], on="pair_id",
            how="inner", validate="one_to_one",
        )
        binary = compared.dropna(subset=["human_label"])
        rated = compared.dropna(subset=["human_score"])
        if binary.empty and rated.empty:
            continue
        agreement, kappa, rho = np.nan, np.nan, np.nan
        if not binary.empty:
            labels = binary["human_label"].astype(int)
            decisions = binary["predicted_label"].astype(int)
            agreement = accuracy_score(labels, decisions)
            if pd.concat([labels, decisions]).nunique() > 1:
                kappa = cohen_kappa_score(labels, decisions, labels=[0, 1])
        if len(rated) >= 2 and rated["score"].nunique() > 1 and rated["human_score"].nunique() > 1:
            rho = float(spearmanr(rated["score"], rated["human_score"]).statistic)
        human_rows.append({
            "model": model, "precision": precision,
            "n_human_labels": len(binary), "agreement": agreement, "cohen_kappa": kappa,
            "n_human_scores": len(rated), "spearman_rho": rho,
        })
    human_metrics = pd.DataFrame(human_rows, columns=human_columns)
    if human_metrics.empty:
        print("Le fichier est présent, mais aucune annotation n'est encore remplie.")
    else:
        display(human_metrics.round(4))

# Un tableau vide indique explicitement l'absence de mesures humaines.
human_metrics.to_csv(RESULTS_DIR / "evaluation_humans.csv", index=False)

## 5. Produire des exemples visuels

Pour chaque configuration disponible, afficher jusqu'à deux réussites et deux erreurs,
choisies avec une graine fixe. Les exemples servent à comprendre les résultats,
pas à estimer leur fréquence : celle-ci est donnée par les mesures sur toutes les paires.

Les images sont reprises depuis Flickr8k avec la même sélection que le notebook 01
(`train`, graine 42, au plus 500 images), puis vérifiées avec
`flickr8k_references.csv`. Cela peut nécessiter un téléchargement si le cache Colab est vide.
En local, des images déjà exportées peuvent être placées dans `data/images/<image_id>.jpg`
(ou `.png`, `.jpeg`). Aucune image n'est ajoutée au dépôt.

In [ ]:
# Charger la sélection Flickr8k une seule fois pour tous les exemples.
visual_sample = None

def get_image(image_id):
    global visual_sample
    image_id = int(image_id)
    for extension in (".jpg", ".png", ".jpeg"):
        path = DATA_DIR / "images" / f"{image_id}{extension}"
        if path.exists():
            with Image.open(path) as image:
                return image.convert("RGB")

    if visual_sample is None:
        from datasets import load_dataset

        reference_columns = [f"caption_{number}" for number in range(5)]
        references = read_csv_checked(
            RESULTS_DIR / "flickr8k_references.csv", ["id"] + reference_columns
        ).set_index("id").sort_index()
        if references.index.tolist() != list(range(len(references))):
            raise ValueError("Les identifiants du notebook 01 ne sont pas ceux attendus.")
        dataset = load_dataset("jxie/flickr8k", split="train")
        size = min(500, len(dataset))
        if len(references) != size or set(pairs["image_id"]) != set(range(size)):
            raise ValueError("La sélection d'images diffère des notebooks précédents.")
        visual_sample = dataset.shuffle(seed=SEED).select(range(size))
        for column in reference_columns:
            actual = [example[column] for example in visual_sample]
            if actual != references[column].tolist():
                raise ValueError(f"Les références {column} ne correspondent pas aux images.")
        print(f"{size} images Flickr8k retrouvées et vérifiées.")
    return visual_sample[image_id]["image"]


example_frames = []
if AFFICHER_IMAGES:
    for (model, precision), frame in experiments.items():
        # Séparer les réussites et les erreurs pour illustrer les deux cas.
        groups = [frame.loc[frame["is_correct"]], frame.loc[~frame["is_correct"]]]
        examples = pd.concat([
            group.sample(n=min(2, len(group)), random_state=SEED) for group in groups
        ])
        example_frames.append(examples)
        fig, axes = plt.subplots(1, len(examples), figsize=(5 * len(examples), 6), squeeze=False)
        for axis, (_, row) in zip(axes.flat, examples.iterrows()):
            axis.imshow(get_image(row["image_id"]))
            status = "Réussite" if row["is_correct"] else "Erreur"
            label = "correcte" if row["label"] else "incorrecte"
            prediction = "correcte" if row["predicted_label"] else "incorrecte"
            title = (
                f"{status} — image {row['image_id']} / paire {row['pair_id']}\n"
                + textwrap.fill(row["caption"], width=40)
                + f"\nLabel automatique : {label}\nDécision : {prediction}"
                + f"\nLoss : {row['loss']:.3f}"
            )
            axis.set_title(title, fontsize=10, color="darkgreen" if row["is_correct"] else "darkred")
            axis.axis("off")
        fig.suptitle(f"{model.upper()} / {precision}", fontsize=14)
        fig.tight_layout()
        fig.savefig(FIGURES_DIR / f"exemples_{model}_{precision}.png", dpi=120, bbox_inches="tight")
        plt.show()
        plt.close(fig)
else:
    print("Affichage des images désactivé ; les mesures et graphiques restent disponibles.")

examples_export = (
    pd.concat(example_frames, ignore_index=True) if example_frames
    else all_predictions.iloc[0:0]
)
examples_export.to_csv(RESULTS_DIR / "evaluation_examples.csv", index=False)

## 6. Lecture des résultats

Comparer d'abord l'accuracy à 50 %, puis les versions 8/4 bits à la référence fp16
de chaque modèle. Une réduction de mémoire peut s'accompagner d'un ralentissement :
le tableau donne les valeurs observées, sans supposer que la quantification accélère toujours.

Les légendes négatives sont obtenues par décalage dans le notebook 02 ; certaines peuvent
tout de même décrire l'image. Les annotations humaines et les exemples d'erreurs permettent
de vérifier cette limite. Ces résultats concernent ce sous-ensemble et cette tâche.

In [ ]:
# Repérer la meilleure accuracy parmi les configurations évaluées.
best = metrics.loc[metrics["accuracy"].idxmax()]
print(
    f"Meilleure accuracy observée : {best['model'].upper()} / {best['precision']} "
    f"= {best['accuracy']:.1%}, sur {int(best['pairs'])} paires."
)
print(f"Configurations évaluées : {len(metrics)}/6 ; baseline théorique : {baseline_accuracy:.0%}.")
print(f"Tableaux enregistrés dans : {RESULTS_DIR}")
print("CSV : evaluation_metrics, evaluation_predictions, evaluation_humans, evaluation_examples.")
print(f"Figures PNG enregistrées dans : {FIGURES_DIR}")
if human_metrics.empty:
    print("Comparaison humaine en attente : compléter human_annotations.csv.")

Les tableaux et figures sont sauvegardés automatiquement sur Drive pour le rapport
et la démonstration. Les CSV de calcul et les images restent hors de Git ;
seul le notebook et, au besoin, une petite sélection de résultats doivent être versionnés.

Références : [mesures de classification (scikit-learn)](https://scikit-learn.org/stable/modules/model_evaluation.html),
[kappa de Cohen](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.cohen_kappa_score.html),
[corrélation de Spearman (SciPy)](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.spearmanr.html).